# Ongoing three-task run: saved fold evidence

This notebook reads progress and hash-verified completed checkpoints. It does not
pool partial folds, compute final metrics, fit models or update the manuscript.
Copy completed checkpoint directories, the root manifest and resolved configuration.
Set `IIVV_RESULTS` to that copied run/snapshot directory.


In [1]:
import json, os
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image, FileLink

REPO = Path.cwd()
if not (REPO / "run_revision.py").is_file():
    REPO = REPO.parent
assert (
    REPO / "run_revision.py"
).is_file(), "Start Jupyter in the repository or manuscript directory"
import sys

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from revision.plot_style import apply_style, color_palette_full

apply_style()

from revision.data import TASK_PROTOCOL, TASK_TITLES, TASK3_CAPTION


def csv(path, limit=None):
    path = Path(path)
    if not path.is_file():
        display(Markdown(f"Pending/unavailable: `{path.name}`"))
        return pd.DataFrame()
    try:
        table = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        table = pd.DataFrame()
    display(table if limit is None else table.head(limit))
    return table


def images(paths):
    for path in paths:
        display(Markdown(f"**{path.name}**"))
        rendered = (
            REPO / "results/revision-palette" / path.relative_to(REPO / "results")
        )
        display(Image(filename=str(rendered if rendered.is_file() else path)))
        if "cancer_vs_lr_hr" in str(path):
            display(Markdown(TASK3_CAPTION.replace("\n", " ")))


from revision.data import digest

ROOT = REPO / os.environ.get("IIVV_RESULTS", "results/revision-three-tasks-v3")
manifest = json.loads((ROOT / "run_manifest.json").read_text())
assert manifest["config"].get("task_protocol") == TASK_PROTOCOL, "Obsolete protocol"
assert not manifest.get("synthetic"), "Synthetic fixture"
display({"status": manifest["status"], "protocol": TASK_PROTOCOL})
completed = []
for marker in sorted((ROOT / "fold-checkpoints").glob("*/*/complete.json")):
    saved = json.loads(marker.read_text())
    assert saved["task"] in manifest["config"]["tasks"]
    for name, expected in saved["artifacts"].items():
        relative = Path(name)
        assert (
            not relative.is_absolute() and ".." not in relative.parts
        ), "Unsafe artifact path"
        assert (
            digest(marker.parent / relative) == expected
        ), f"Changed checkpoint: {marker.parent / relative}"
    classes = json.loads((marker.parent / "classes.json").read_text())
    assert classes["task_protocol"] == TASK_PROTOCOL
    completed.append(
        {
            "task": saved["task"],
            "repeat": saved["fold_key"][0],
            "fold": saved["fold_key"][1],
            "seconds": saved["elapsed_seconds"],
            "folder": str(marker.parent),
        }
    )
display(pd.DataFrame(completed))
print(
    f"Completed outer jobs: {len(completed)}/{len(manifest['config']['tasks']) * manifest['config']['outer_folds'] * manifest['config']['outer_repeats']}"
)

{'status': 'completed', 'protocol': 'three-tasks-v1-lesion-only-task3'}

,task,repeat,fold,seconds,folder
0,cancer_vs_lr_hr,0,0,1225.539351,/home/jacob/Schreibtisch/Uni/Forschung/IIVV-Ev...
1,cancer_vs_lr_hr,0,1,5780.710059,/home/jacob/Schreibtisch/Uni/Forschung/IIVV-Ev...
2,cancer_vs_lr_hr,0,2,1614.018572,/home/jacob/Schreibtisch/Uni/Forschung/IIVV-Ev...
3,cancer_vs_lr_hr,0,3,4232.257582,/home/jacob/Schreibtisch/Uni/Forschung/IIVV-Ev...
4,cancer_vs_lr_hr,0,4,4220.427745,/home/jacob/Schreibtisch/Uni/Forschung/IIVV-Ev...
5,healthy_vs_cancer,0,0,385.848226,/home/jacob/Schreibtisch/Uni/Forschung/IIVV-Ev...
6,healthy_vs_cancer,0,1,584.486996,/home/jacob/Schreibtisch/Uni/Forschung/IIVV-Ev...
7,healthy_vs_cancer,0,2,370.023116,/home/jacob/Schreibtisch/Uni/Forschung/IIVV-Ev...
8,healthy_vs_cancer,0,3,5303.384268,/home/jacob/Schreibtisch/Uni/Forschung/IIVV-Ev...
9,healthy_vs_cancer,0,4,615.878209,/home/jacob/Schreibtisch/Uni/Forschung/IIVV-Ev...


Completed outer jobs: 15/15


## Training-selected combinations and candidate failures in completed folds

In [2]:
for row in completed:
    folder = Path(row["folder"])
    display(Markdown(f"### {row['task']}, fold {row['fold']}"))
    display(pd.json_normalize(json.loads((folder / "selections.json").read_text())))
    display(pd.DataFrame(json.loads((folder / "candidate_failures.json").read_text())))

### cancer_vs_lr_hr, fold 0

,task,stream,objective,model,repeat,outer_fold,evaluation,inner_selection_objective,inner_fit_provenance,calibration,...,candidate.selector,candidate.representation,candidate.cutoff,candidate.normalization,candidate.imputation,candidate.class_weight,candidate.patient_weighted,candidate.params_json,source_version.module,source_version.sha256
0,cancer_vs_lr_hr,baseline_svm,baseline,svm,0,0,nested_cv,NaN,"[{'train_patients': ['0', '1', '11', '13', '14...",True,...,all,magnitude,60.000000,freq,mean,None,True,"{""max_iter"": 100000}",NaN,NaN
1,cancer_vs_lr_hr,baseline_rf,baseline,rf,0,0,nested_cv,NaN,"[{'train_patients': ['0', '1', '11', '13', '14...",True,...,all,magnitude,60.000000,freq,mean,None,True,{},NaN,NaN
2,cancer_vs_lr_hr,baseline_lr,baseline,lr,0,0,nested_cv,NaN,"[{'train_patients': ['0', '1', '11', '13', '14...",True,...,all,magnitude,60.000000,freq,mean,None,True,{},NaN,NaN
3,cancer_vs_lr_hr,ablation_selector_pca,ablation,svm,0,0,nested_cv,NaN,"[{'train_patients': ['0', '1', '11', '13', '14...",True,...,pca,magnitude,100.000001,global,mean,balanced,True,"{""max_iter"": 100000}",NaN,NaN
4,cancer_vs_lr_hr,ablation_selector_mutual_info,ablation,svm,0,0,nested_cv,NaN,"[{'train_patients': ['0', '1', '11', '13', '14...",True,...,mutual_info,magnitude,100.000001,global,mean,balanced,True,"{""max_iter"": 100000}",NaN,NaN
5,cancer_vs_lr_hr,ablation_representation_magnitude,ablation,svm,0,0,nested_cv,NaN,"[{'train_patients': ['0', '1', '11', '13', '14...",True,...,pca,magnitude,100.000001,global,mean,balanced,True,"{""max_iter"": 100000}",NaN,NaN
6,cancer_vs_lr_hr,ablation_representation_magnitude_phase,ablation,svm,0,0,nested_cv,NaN,"[{'train_patients': ['0', '1', '11', '13', '14...",True,...,pca,magnitude_phase,100.000001,global,mean,balanced,True,"{""max_iter"": 100000}",NaN,NaN
7,cancer_vs_lr_hr,ablation_quality_80,ablation,svm,0,0,nested_cv,NaN,"[{'train_patients': ['0', '1', '11', '13', '14...",True,...,pca,magnitude,80.000000,global,mean,balanced,True,"{""max_iter"": 100000}",NaN,NaN
8,cancer_vs_lr_hr,ablation_quality_100.000001,ablation,svm,0,0,nested_cv,NaN,"[{'train_patients': ['0', '1', '11', '13', '14...",True,...,pca,magnitude,100.000001,global,mean,balanced,True,"{""max_iter"": 100000}",NaN,NaN
9,cancer_vs_lr_hr,ablation_normalization_global,ablation,svm,0,0,nested_cv,NaN,"[{'train_patients': ['0', '1', '11', '13', '14...",True,...,pca,magnitude,100.000001,global,mean,balanced,True,"{""max_iter"": 100000}",NaN,NaN


""


### cancer_vs_lr_hr, fold 1

,task,stream,objective,model,repeat,outer_fold,evaluation,inner_selection_objective,inner_fit_provenance,calibration,...,candidate.selector,candidate.representation,candidate.cutoff,candidate.normalization,candidate.imputation,candidate.class_weight,candidate.patient_weighted,candidate.params_json,source_version.module,source_version.sha256
0,cancer_vs_lr_hr,baseline_svm,baseline,svm,0,1,nested_cv,NaN,"[{'train_patients': ['1', '15', '19', '2', '20...",True,...,all,magnitude,60.000000,freq,mean,None,True,"{""max_iter"": 100000}",NaN,NaN
1,cancer_vs_lr_hr,baseline_rf,baseline,rf,0,1,nested_cv,NaN,"[{'train_patients': ['1', '15', '19', '2', '20...",True,...,all,magnitude,60.000000,freq,mean,None,True,{},NaN,NaN
2,cancer_vs_lr_hr,baseline_lr,baseline,lr,0,1,nested_cv,NaN,"[{'train_patients': ['1', '15', '19', '2', '20...",True,...,all,magnitude,60.000000,freq,mean,None,True,{},NaN,NaN
3,cancer_vs_lr_hr,ablation_selector_pca,ablation,svm,0,1,nested_cv,NaN,"[{'train_patients': ['1', '15', '19', '2', '20...",True,...,pca,magnitude,100.000001,global,mean,balanced,True,"{""max_iter"": 100000}",NaN,NaN
4,cancer_vs_lr_hr,ablation_selector_mutual_info,ablation,svm,0,1,nested_cv,NaN,"[{'train_patients': ['1', '15', '19', '2', '20...",True,...,mutual_info,magnitude,100.000001,global,mean,balanced,True,"{""max_iter"": 100000}",NaN,NaN
5,cancer_vs_lr_hr,ablation_representation_magnitude,ablation,svm,0,1,nested_cv,NaN,"[{'train_patients': ['1', '15', '19', '2', '20...",True,...,pca,magnitude,100.000001,global,mean,balanced,True,"{""max_iter"": 100000}",NaN,NaN
6,cancer_vs_lr_hr,ablation_representation_magnitude_phase,ablation,svm,0,1,nested_cv,NaN,"[{'train_patients': ['1', '15', '19', '2', '20...",True,...,pca,magnitude_phase,100.000001,global,mean,balanced,True,"{""max_iter"": 100000}",NaN,NaN
7,cancer_vs_lr_hr,ablation_quality_80,ablation,svm,0,1,nested_cv,NaN,"[{'train_patients': ['1', '15', '19', '2', '20...",True,...,pca,magnitude,80.000000,global,mean,balanced,True,"{""max_iter"": 100000}",NaN,NaN
8,cancer_vs_lr_hr,ablation_quality_100.000001,ablation,svm,0,1,nested_cv,NaN,"[{'train_patients': ['1', '15', '19', '2', '20...",True,...,pca,magnitude,100.000001,global,mean,balanced,True,"{""max_iter"": 100000}",NaN,NaN
9,cancer_vs_lr_hr,ablation_normalization_global,ablation,svm,0,1,nested_cv,NaN,"[{'train_patients': ['1', '15', '19', '2', '20...",True,...,pca,magnitude,100.000001,global,mean,balanced,True,"{""max_iter"": 100000}",NaN,NaN


""


... (35 outputs left out)

""


### highrisk_vs_lowrisk, fold 4

,task,stream,objective,model,repeat,outer_fold,evaluation,inner_selection_objective,inner_fit_provenance,calibration,...,candidate.normalization,candidate.imputation,candidate.class_weight,candidate.patient_weighted,candidate.params_json,specificity_thresholds.0.8,specificity_thresholds.0.9,specificity_thresholds.0.95,source_version.module,source_version.sha256
0,highrisk_vs_lowrisk,baseline_svm,baseline,svm,0,4,nested_cv,NaN,"[{'train_patients': ['10', '12', '13', '15', '...",True,...,freq,mean,None,True,"{""max_iter"": 100000}",0.537605,0.722037,0.789883,NaN,NaN
1,highrisk_vs_lowrisk,baseline_rf,baseline,rf,0,4,nested_cv,NaN,"[{'train_patients': ['10', '12', '13', '15', '...",True,...,freq,mean,None,True,{},0.531357,0.609803,0.763384,NaN,NaN
2,highrisk_vs_lowrisk,baseline_lr,baseline,lr,0,4,nested_cv,NaN,"[{'train_patients': ['10', '12', '13', '15', '...",True,...,freq,mean,None,True,{},0.546342,0.646328,0.846037,NaN,NaN
3,highrisk_vs_lowrisk,ablation_selector_pca,ablation,svm,0,4,nested_cv,NaN,"[{'train_patients': ['10', '12', '13', '15', '...",True,...,global,mean,balanced,True,"{""max_iter"": 100000}",0.476258,0.739287,0.814288,NaN,NaN
4,highrisk_vs_lowrisk,ablation_selector_mutual_info,ablation,svm,0,4,nested_cv,NaN,"[{'train_patients': ['10', '12', '13', '15', '...",True,...,global,mean,balanced,True,"{""max_iter"": 100000}",0.489824,0.733043,0.811490,NaN,NaN
5,highrisk_vs_lowrisk,ablation_representation_magnitude,ablation,svm,0,4,nested_cv,NaN,"[{'train_patients': ['10', '12', '13', '15', '...",True,...,global,mean,balanced,True,"{""max_iter"": 100000}",0.476258,0.739287,0.814288,NaN,NaN
6,highrisk_vs_lowrisk,ablation_representation_magnitude_phase,ablation,svm,0,4,nested_cv,NaN,"[{'train_patients': ['10', '12', '13', '15', '...",True,...,global,mean,balanced,True,"{""max_iter"": 100000}",0.493174,0.718964,0.805418,NaN,NaN
7,highrisk_vs_lowrisk,ablation_quality_80,ablation,svm,0,4,nested_cv,NaN,"[{'train_patients': ['10', '12', '13', '15', '...",True,...,global,mean,balanced,True,"{""max_iter"": 100000}",0.495793,0.731651,0.807008,NaN,NaN
8,highrisk_vs_lowrisk,ablation_quality_100.000001,ablation,svm,0,4,nested_cv,NaN,"[{'train_patients': ['10', '12', '13', '15', '...",True,...,global,mean,balanced,True,"{""max_iter"": 100000}",0.491856,0.729428,0.807651,NaN,NaN
9,highrisk_vs_lowrisk,ablation_normalization_global,ablation,svm,0,4,nested_cv,NaN,"[{'train_patients': ['10', '12', '13', '15', '...",True,...,global,mean,balanced,True,"{""max_iter"": 100000}",0.491856,0.729428,0.807651,NaN,NaN


""


## Saved progress from the running folds

In [3]:
progress = []
for path in sorted((ROOT / "fold-checkpoints").glob("*/*/progress.json")):
    if not (path.parent / "complete.json").exists():
        progress.append(dict(folder=str(path.parent), **json.loads(path.read_text())))
display(pd.json_normalize(progress))

""
